In [ ]:
!pip install pyhealth torch scikit-learn pandas tqdm

In [ ]:
!pip list | grep pyhealth

In [ ]:
from pyhealth.utils import set_seed

set_seed(42)

In [ ]:
import tempfile
from pyhealth.datasets import MIMIC3Dataset

In [ ]:
base_dataset = MIMIC3Dataset(
    root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    cache_dir=tempfile.mkdtemp(),
    dev=True,
)

base_dataset.stats()

In [ ]:
from pyhealth.tasks import ReadmissionPredictionMIMIC3

task = ReadmissionPredictionMIMIC3(exclude_minors=False)
sample_dataset = base_dataset.set_task(task)

In [ ]:
from pyhealth.datasets import split_by_patient, get_dataloader

train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
from pyhealth.models import RNN

model = RNN(dataset=sample_dataset)

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(model=model)
trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=1,
    monitor="roc_auc",
)

In [ ]:
trainer.evaluate(test_dataloader)

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset, split_by_patient, get_dataloader
from pyhealth.tasks import ReadmissionPredictionMIMIC3
from pyhealth.models import RNN
from pyhealth.trainer import Trainer


if __name__ == "__main__":

    # =========================
    # STEP 1: Load dataset
    # =========================
    cache_dir = tempfile.mkdtemp()

    base_dataset = MIMIC3Dataset(
        root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=cache_dir,
        dev=False,   # FIX: use full dataset for stable class distribution
    )

    base_dataset.stats()


    # =========================
    # STEP 2: Define task
    # =========================
    task = ReadmissionPredictionMIMIC3(exclude_minors=False)
    sample_dataset = base_dataset.set_task(task)


    # =========================
    # STEP 3: Train / Val / Test split
    # =========================
    train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset,
        [0.8, 0.1, 0.1],
        seed=42   # FIX: reproducible and more stable splits
    )

    train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
    val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
    test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)


    # =========================
    # STEP 4: Define model
    # =========================
    model = RNN(
        dataset=sample_dataset
    )


    # =========================
    # STEP 5: Train model
    # =========================
    trainer = Trainer(model=model)

    trainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=5,
        monitor="roc_auc",   # required metric
    )


    # =========================
    # STEP 6: Evaluate model
    # =========================
    results = trainer.evaluate(test_dataloader)

    print("\nFinal Test Results:")
    print(results)